# ❄️ Snowflake Time Travel & Fail-safe — Detailed Notes with Commands & Examples

---

# 🧠 1. What is Time Travel?

👉 Time Travel allows you to **access historical data** (past versions of tables)

---

## 💡 What You Can Do

- Recover deleted data
- Undo updates
- Query past state

---

## ⏱️ Retention Period

- Permanent tables → 1 to 90 days
- Transient tables → 0 to 1 day
- Temporary tables → Not supported

---

# ⚙️ 2. Basic Example

## Step 1: Create Table

CREATE TABLE orders (
  id INT,
  amount NUMBER
);

---

## Step 2: Insert Data

INSERT INTO orders VALUES (1,100);

---

## Step 3: Delete Data

DELETE FROM orders;

---

## Step 4: Recover Using Time Travel

SELECT * FROM orders AT (OFFSET => -60*5);

👉 Gets data from 5 minutes ago

---

# ⚙️ 3. Using Timestamp

SELECT * FROM orders
AT (TIMESTAMP => '2024-01-01 10:00:00');

---

# ⚙️ 4. Using BEFORE

SELECT * FROM orders
BEFORE (STATEMENT => 'query_id');

---

# 🔄 5. Restore Table

CREATE TABLE orders_restore CLONE orders
AT (OFFSET => -3600);

---

# 🔁 6. Undrop Table

DROP TABLE orders;

UNDROP TABLE orders;

---

# 🔐 7. What is Fail-safe?

👉 Fail-safe is a **7-day backup period after Time Travel expires**

---

## 💡 Key Points

- Fixed duration: 7 days
- Not configurable
- Only Snowflake support can restore

---

# 🔄 8. Lifecycle

Active Data
↓
Time Travel
↓
Fail-safe
↓
Permanent Delete

---

# 🧠 9. Real Example

Day 1 → Data created
Day 5 → Data deleted

If retention = 7 days:

- Day 6 → Recover via Time Travel
- Day 10 → Recover via Fail-safe
- Day 13 → Data permanently lost

---

# ⚖️ 10. Difference

| Feature | Time Travel | Fail-safe |
|--------|------------|----------|
| Access | User | Snowflake support |
| Duration | 1–90 days | Fixed 7 days |
| Control | Yes | No |

---

# ⚡ 11. Important Commands

-- Check retention
SHOW PARAMETERS LIKE 'DATA_RETENTION_TIME_IN_DAYS';

-- Set retention
ALTER TABLE orders SET DATA_RETENTION_TIME_IN_DAYS = 7;

---

# 🚨 12. Important Points

- Time Travel = self-service recovery
- Fail-safe = last backup
- Transient → No fail-safe
- Temporary → No recovery

---

# 🎯 ONE LINE

Time Travel = user recovery
Fail-safe = Snowflake backup


# ❄️ Snowflake Time Travel & Fail-safe — Internal Working (Deep Dive)

---

# 🧠 1. Core Idea

Snowflake does NOT store multiple copies of data.

👉 Uses:
- Immutable Micro-Partitions
- Metadata Versioning

---

# 🧱 2. Micro-Partitions

- ~16MB chunks
- Immutable
- Columnar

👉 Data is never updated, only new versions created

---

# 🔄 3. Update Internals

Example:
UPDATE orders SET amount = 200 WHERE id = 1;

Internally:
- Old partition kept
- New partition created
- Metadata updated

---

# 🧠 4. Time Travel

👉 Uses old metadata snapshots

Example:
SELECT * FROM orders AT (OFFSET => -3600);

---

# 🧠 5. Snapshots

T1 → old data
T2 → new data

Time Travel = access T1

---

# 🔐 6. Lifecycle

Active → Time Travel → Fail-safe → Delete

---

# 🔒 7. Fail-safe

- 7 days fixed
- Only Snowflake support can access
- Metadata hidden

---

# ⚡ 8. Copy-on-Write

- Updates create new data
- Old data retained

---

# 💰 9. Cost

- Old partitions stored → cost

---

# 🚨 10. Key Points

- No overwrite
- Metadata controls data
- Time Travel = rewind
- Fail-safe = backup

---

# 🎯 One Line

Time Travel = metadata rewind
Fail-safe = hidden backup
